In [24]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/melting_point_project"   # поправьте под своё название папки
RAW = f"{BASE}/data/raw"
PROC = f"{BASE}/data/processed"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

In [3]:
X = pd.read_parquet(f"{PROC}/descriptors.parquet")      # или pd.read_csv(...)
meta = pd.read_csv(f"{PROC}/melting_points_clean.csv")
y = meta["mpC"].values
print(X.shape, len(y))

(19824, 215) 19824


In [4]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.dummy import DummyRegressor

In [5]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 37.2 MB/s  0:00:00


In [6]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit

In [8]:
scaf = meta['smiles_can'].apply(lambda s: MurckoScaffold.MurckoScaffoldSmiles(smiles=s))
groups = [sc if sc else s for sc, s in zip(scaf, meta['smiles_can'])]

Create split with 5 folds taking into account groups

In [14]:
splits = list(GroupKFold(n_splits=5).split(X, y, groups))

In [16]:
splits


[(array([    0,     1,     2, ..., 19821, 19822, 19823], shape=(14495,)),
  array([   12,    16,    17, ..., 19332, 19333, 19782], shape=(5329,))),
 (array([    0,     1,     2, ..., 19820, 19822, 19823], shape=(16200,)),
  array([    3,     5,    13, ..., 19811, 19817, 19821], shape=(3624,))),
 (array([    1,     2,     3, ..., 19818, 19821, 19823], shape=(16200,)),
  array([    0,     4,    14, ..., 19819, 19820, 19822], shape=(3624,))),
 (array([    0,     2,     3, ..., 19821, 19822, 19823], shape=(16200,)),
  array([    1,     7,     9, ..., 19813, 19815, 19818], shape=(3624,))),
 (array([    0,     1,     3, ..., 19820, 19821, 19822], shape=(16201,)),
  array([    2,     6,     8, ..., 19814, 19816, 19823], shape=(3623,)))]

In [15]:
print("размеры тестовых частей:", [len(te) for tr, te in splits])

размеры тестовых частей: [5329, 3624, 3624, 3624, 3623]


SimpleImputer(strategy="median") заполняет пропуски в таблице X.

In [18]:
models = {
    "median": DummyRegressor(strategy="median"),
    "rf": make_pipeline(SimpleImputer(strategy="median"),
                        RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=0)),
}

In [19]:
scoring = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error"}

**Check differencies between **bold text** all descriptors, mine choise and choise of Claude**

In [20]:
rows = []
for name, model in models.items():
    r = cross_validate(model, X, y, cv=splits, scoring=scoring)
    rows.append({
        "model": name,
        "MAE": -r["test_MAE"].mean(),   "MAE_std": r["test_MAE"].std(),
        "RMSE": -r["test_RMSE"].mean(), "RMSE_std": r["test_RMSE"].std(),
    })
    print(name, "RMSE по фолдам:", np.round(-r["test_RMSE"], 1))

print(pd.DataFrame(rows).round(1))

median RMSE по фолдам: [ 74.2  96.6  97.   98.6 100.1]
rf RMSE по фолдам: [48.6 48.4 46.5 44.9 47.2]
    model   MAE  MAE_std  RMSE  RMSE_std
0  median  74.7      6.8  93.3       9.6
1      rf  35.7      1.9  47.1       1.4


In [22]:
from sklearn.feature_selection import SelectFromModel

K = 11

rf_top = make_pipeline(
    SimpleImputer(strategy="median"),
    SelectFromModel(RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=0, max_features=0.3),
    max_features=K, threshold=-np.inf),
    RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=0),
)

In [23]:
claude_cols = ["MolWt", "HeavyAtomCount", "NumHDonors", "NumHAcceptors", "TPSA",
             "MolLogP", "NumRotatableBonds", "FractionCSP3",
             "NumAromaticRings", "RingCount", "SymmetryRatio"]

In [ ]:
mine_cols = [...]

In [ ]:
configs = {
    "0. все дескрипторы":  (models["rf"], X),
    "1. топ-K по модели":  (rf_top,       X),
    "2. ваш набор":        (models["rf"], X[mine_cols]),
    "3. мой набор":        (models["rf"], X[claude_cols]),
}

rows = []
for name, (m, Xs) in configs.items():
    t = time.time()
    r = cross_validate(m, Xs, y, cv=splits, scoring=scoring)
    rows.append({"набор": name,
                 "MAE": -r["test_MAE"].mean(),
                 "RMSE": -r["test_RMSE"].mean(), "RMSE_std": r["test_RMSE"].std(),
                 "время, с": time.time() - t})
print(pd.DataFrame(rows).round(1))